# Choro Songbook Corpus (DCMLab/choro v1.3.3) to TiLiA

Full run: download the corpus's zip (checked by md5), convert the 295 pieces, run TiLiA, build the package. Output directory: `$DATASET_CONVERTERS_OUT`. Needs `TILIA` pointing at TiLiA's command line and format-converters installed.

In [ ]:
import importlib.util, json, os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "common").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
spec = importlib.util.spec_from_file_location("choro_convert", root / "choro" / "convert.py")
convert = importlib.util.module_from_spec(spec)
spec.loader.exec_module(convert)

out = Path(os.environ["DATASET_CONVERTERS_OUT"])
out.mkdir(parents=True, exist_ok=True)

## Download

One zip from Zenodo; `fetch` rejects an md5 mismatch.

In [ ]:
pieces = convert.read_corpus(convert.fetch_corpus())
print(len(pieces), "pieces,", sum(len(p.rows) for p in pieces.values()), "rows")

## Convert and run TiLiA

In [ ]:
summary = convert.convert(list(pieces), out, pieces=pieces, jobs=6)

## Package

In [ ]:
print(convert.package(out, out / "package"))

## Summary

Components in the saved files against rows counted from the source table, and what `source-report.json` lists.

In [ ]:
totals = {}
for layers in summary.values():
    for layer, v in layers.items():
        t = totals.setdefault(layer, [0, 0])
        t[0] += v["components"]; t[1] += v["source_rows"]
print(len(summary), "pieces")
for layer, (comps, rows) in totals.items():
    print(f"{layer:16} components {comps:6}  source rows {rows:6}")
assert all(v["components"] == v["source_rows"] for l in summary.values() for v in l.values())

report = json.loads((out / "source-report.json").read_text())
print(sum(1 for r in report.values() if r["irregular_bars"]), "pieces with bars that don't add up to the time signature")
print(sum(len(r["problems"]) for r in report.values()), "corrections and form notes")